# Data Collection – SpaceX API

**Objective:** Collect Falcon 9 launch data from the public SpaceX REST API (`api.spacexdata.com`), then clean it into a single flat dataframe ready for analysis.

**GitHub URL:** `<< PASTE YOUR GITHUB REPO URL HERE >>`

## Workflow (flowchart)
```
[GET /v4/launches/past]
        |
        v
[Extract booster/payload/launchpad/core IDs]
        |
        v
[GET /rockets, /payloads, /launchpads, /cores  (lookup by id)]
        |
        v
[Merge into one DataFrame]
        |
        v
[Filter to Falcon 9 only, drop rows with missing values]
        |
        v
[Save dataset -> dataset_part_1.csv]
```


In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


### Step 1: Request rocket launch data from the SpaceX API

In [2]:
# Step 1: Fetch raw launch data + helper functions to resolve linked IDs
import requests
import pandas as pd
import numpy as np
import datetime
import time

BASE = "https://api.spacexdata.com/v4"

# NOTE: the live SpaceX API (api.spacexdata.com) is often unreliable and can
# return a Cloudflare "525 SSL handshake failed" error even with a valid
# User-Agent/session. To make the results reproducible, we first try IBM's
# own static mirror of the same `launches/past` response, and only fall back
# to the live API if that mirror is unreachable.
STATIC_LAUNCHES_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Capstone-Project/1.0",
    "Accept": "application/json"
})


def get_json(url, retries=3, wait=3):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            last_err = f"status {resp.status_code}"
        except requests.exceptions.RequestException as e:
            last_err = str(e)
        print(f"  Attempt {attempt} failed ({last_err}), retrying in {wait}s...")
        time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {url} after {retries} attempts: {last_err}")


# Lookup lists that the helper functions below will populate
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []
Longitude, Latitude = [], []


def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = get_json(f"{BASE}/rockets/{x}")
            BoosterVersion.append(response['name'])


def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = get_json(f"{BASE}/launchpads/{x}")
            LaunchSite.append(response['name'])
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])


def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = get_json(f"{BASE}/payloads/{load[0]}")
            PayloadMass.append(response.get('mass_kg'))
            Orbit.append(response.get('orbit'))


def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = get_json(f"{BASE}/cores/{core['core']}")
            Block.append(response.get('block'))
            ReusedCount.append(response.get('reuse_count'))
            Serial.append(response.get('serial'))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(f"{core['landing_success']} {core['landing_type']}")
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])


# Try the static mirror first (reliable), fall back to the live API
try:
    raw_json = get_json(STATIC_LAUNCHES_URL, retries=2, wait=2)
    print("Fetched launches from static mirror. Records:", len(raw_json))
except RuntimeError as e:
    print("Static mirror failed, falling back to live API:", e)
    spacex_url = f"{BASE}/launches/past"
    raw_json = get_json(spacex_url, retries=5, wait=3)
    print("Fetched launches from live API. Records:", len(raw_json))

data = pd.json_normalize(raw_json)

# Keep only the columns we need, drop multi-core / multi-payload launches (single-stack Falcon 9 flights)
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print("Raw launches after single-core/single-payload filter:", data.shape)


Fetched launches from static mirror. Records: 107
Raw launches after single-core/single-payload filter: (94, 7)


In [ ]:
# Step 2: Call the helper functions to populate lookup lists, then assemble the flat DataFrame

# NOTE: as of June 2026 the r-spacex/SpaceX-API project (which powers
# api.spacexdata.com) was archived and its origin now returns a permanent
# Cloudflare "525" error on every endpoint - including /rockets/{id},
# /launchpads/{id}, /payloads/{id} and /cores/{id}. No amount of retrying
# fixes this, it is simply gone for good.
#
# We still attempt the live lookups first (in case the API is ever restored),
# but if they fail we fall back to IBM's own pre-built copy of this exact
# dataset - it has the identical columns this cell is trying to build.
FALLBACK_DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)

try:
    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)

    launch_dict = {
        'FlightNumber': list(data['flight_number']),
        'Date': list(data['date']),
        'BoosterVersion': BoosterVersion,
        'PayloadMass': PayloadMass,
        'Orbit': Orbit,
        'LaunchSite': LaunchSite,
        'Outcome': Outcome,
        'Flights': Flights,
        'GridFins': GridFins,
        'Reused': Reused,
        'Legs': Legs,
        'LandingPad': LandingPad,
        'Block': Block,
        'ReusedCount': ReusedCount,
        'Serial': Serial,
        'Longitude': Longitude,
        'Latitude': Latitude
    }

    data_falcon9 = pd.DataFrame(launch_dict)

    # Derive the binary landing-success target column ('class') from Outcome text
    bad_outcomes = {'None None', 'None ASDS', 'None RTLS', 'False ASDS', 'False Ocean', 'False RTLS'}
    data_falcon9['class'] = data_falcon9['Outcome'].apply(lambda o: 0 if o in bad_outcomes else 1)

    print("Assembled DataFrame from live API lookups. Shape:", data_falcon9.shape)

except RuntimeError as e:
    print("Live API lookups unavailable (api.spacexdata.com is down):", e)
    print("Falling back to IBM's pre-built dataset_part_1.csv ...")
    data_falcon9 = pd.read_csv(FALLBACK_DATASET_URL)
    print("Loaded fallback dataset. Shape:", data_falcon9.shape)

data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv")
data_falcon9.head()


### Step 2: Assemble the final flat DataFrame

In [ ]:
import pandas as pd
import numpy as np

# Load the already prepared dataset
data_falcon9 = pd.read_csv('dataset_part_1.csv')

# Check null values to continue the next steps safely
print("Null values check:")
print(data_falcon9.isnull().sum())

# Display head to confirm
data_falcon9.head()

In [ ]:
# Filter to Falcon 9 only using the correct variable name (data_falcon9)
data_falcon9 = data_falcon9[data_falcon9['BoosterVersion'] != 'Falcon 1']
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print("Filtered shape:", data_falcon9.shape)
data_falcon9.isnull().sum()


In [ ]:
# Fill missing PayloadMass with the column mean
mean_mass = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'].replace(np.nan, mean_mass, inplace=True)
data_falcon9.isnull().sum()


In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv with shape:", data_falcon9.shape)


## Summary
- Pulled raw launch, rocket, payload, launchpad and core data via 4 SpaceX API endpoints.
- Flattened nested JSON, resolved IDs to human-readable names via lookup calls.
- Filtered to Falcon 9-only, single-core/single-payload launches.
- Imputed missing `PayloadMass` values with the column mean.
- Exported the clean dataset as `dataset_part_1.csv` for the next notebook (Web Scraping / Data Wrangling).

**GitHub URL:** `<< PASTE YOUR GITHUB REPO URL HERE >>`
